# Holiday Package Prediction using. Multiple Models and implementating Random Forest

## Holiday Package Prediciton

### 1) Problem statement.
"Trips & Travel.Com" company wants to enable and establish a viable business model to expand the customer base.
One of the ways to expand the customer base is to introduce a new offering of packages. Currently, there are 5 types of packages the company is offering * Basic, Standard, Deluxe, Super Deluxe, King. Looking at the data of the last year, we observed that 18% of the customers purchased the packages. However, the marketing cost was quite high because customers were contacted at random without looking at the available information.
The company is now planning to launch a new product i.e. Wellness Tourism Package. Wellness Tourism is defined as Travel that allows the traveler to maintain, enhance or kick-start a healthy lifestyle, and support or increase one's sense of well-being.
However, this time company wants to harness the available data of existing and potential customers to make the marketing expenditure more efficient.
### 2) Data Collection.
The Dataset is collected from https://www.kaggle.com/datasets/susant4learning/holiday-package-purchase-prediction
The data consists of 20 column and 4888 rows.


In [103]:
# import all the necessary librariess
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px

from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold,cross_val_predict

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [57]:
# Load the dataset
dataset=pd.read_csv('dataset/Travel.csv')

In [58]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
2334,202334,1,41.0,Company Invited,1,11.0,Salaried,Male,3,4.0,Basic,5.0,Married,7.0,0,3,0,1.0,Executive,17107.0
2072,202072,0,30.0,Self Enquiry,1,8.0,Small Business,Female,2,4.0,Deluxe,3.0,Unmarried,6.0,1,1,1,1.0,Manager,21877.0
39,200039,0,33.0,Company Invited,3,6.0,Salaried,Female,2,2.0,Deluxe,5.0,Divorced,3.0,0,3,1,0.0,Manager,20376.0
3087,203087,0,43.0,Self Enquiry,1,10.0,Salaried,Female,4,2.0,Deluxe,3.0,Divorced,4.0,1,5,0,1.0,Manager,23909.0
4229,204229,0,44.0,Company Invited,1,17.0,Salaried,Female,3,4.0,Basic,3.0,Single,2.0,0,5,0,1.0,Executive,21133.0


In [59]:
# Data Cleaning
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 20 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   CustomerID                4888 non-null   int64  
 1   ProdTaken                 4888 non-null   int64  
 2   Age                       4662 non-null   float64
 3   TypeofContact             4863 non-null   object 
 4   CityTier                  4888 non-null   int64  
 5   DurationOfPitch           4637 non-null   float64
 6   Occupation                4888 non-null   object 
 7   Gender                    4888 non-null   object 
 8   NumberOfPersonVisiting    4888 non-null   int64  
 9   NumberOfFollowups         4843 non-null   float64
 10  ProductPitched            4888 non-null   object 
 11  PreferredPropertyStar     4862 non-null   float64
 12  MaritalStatus             4888 non-null   object 
 13  NumberOfTrips             4748 non-null   float64
 14  Passport

In [60]:
dataset.isna().sum()

CustomerID                    0
ProdTaken                     0
Age                         226
TypeofContact                25
CityTier                      0
DurationOfPitch             251
Occupation                    0
Gender                        0
NumberOfPersonVisiting        0
NumberOfFollowups            45
ProductPitched                0
PreferredPropertyStar        26
MaritalStatus                 0
NumberOfTrips               140
Passport                      0
PitchSatisfactionScore        0
OwnCar                        0
NumberOfChildrenVisiting     66
Designation                   0
MonthlyIncome               233
dtype: int64

In [61]:
dataset[dataset.isna().any(axis=1)]

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
4,200004,0,NaN,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
11,200011,0,NaN,Self Enquiry,1,21.0,Salaried,Female,2,4.0,Deluxe,3.0,Single,1.0,1,3,0,0.0,Manager,NaN
19,200019,0,NaN,Self Enquiry,1,8.0,Salaried,Male,2,3.0,Basic,3.0,Single,6.0,1,4,0,1.0,Executive,NaN
20,200020,0,NaN,Company Invited,1,17.0,Salaried,Female,3,2.0,Deluxe,3.0,Married,1.0,0,3,1,2.0,Manager,NaN
21,200021,1,NaN,Self Enquiry,3,15.0,Salaried,Male,2,4.0,Deluxe,5.0,Single,1.0,0,2,0,0.0,Manager,18407.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4850,204850,1,46.0,Self Enquiry,3,8.0,Salaried,Male,4,5.0,Deluxe,5.0,Married,NaN,0,4,1,3.0,Manager,36739.0
4851,204851,1,40.0,Self Enquiry,1,9.0,Salaried,Female,4,4.0,Basic,5.0,Married,NaN,1,1,1,1.0,Executive,35801.0
4868,204868,1,43.0,Company Invited,2,15.0,Salaried,Female,4,5.0,Basic,3.0,Married,NaN,0,5,1,2.0,Executive,36539.0
4869,204869,1,56.0,Self Enquiry,3,16.0,Small Business,Female,3,6.0,Basic,4.0,Single,NaN,0,1,1,2.0,Executive,37865.0


In [62]:
dataset.duplicated().sum()

np.int64(0)

In [63]:
# Split to categorical and numerical columns
categorical_cols=dataset.select_dtypes(include='O').columns
numerical_cols=dataset.select_dtypes(exclude='O').columns

In [64]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
2667,202667,0,39.0,Self Enquiry,3,10.0,Salaried,Male,4,4.0,Standard,3.0,Married,2.0,0,2,1,2.0,Senior Manager,29287.0
4278,204278,0,38.0,Self Enquiry,1,17.0,Small Business,Female,4,4.0,Basic,5.0,Married,8.0,0,1,1,3.0,Executive,22130.0
2695,202695,0,40.0,Company Invited,1,9.0,Large Business,Female,4,4.0,Standard,3.0,Single,2.0,0,2,1,2.0,Senior Manager,29616.0
3525,203525,0,36.0,Self Enquiry,1,23.0,Salaried,Fe Male,4,5.0,Standard,4.0,Unmarried,3.0,0,1,1,3.0,Senior Manager,27284.0
759,200759,0,45.0,Self Enquiry,2,30.0,Small Business,Male,2,3.0,Basic,4.0,Single,2.0,0,4,0,0.0,Executive,17177.0


In [65]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [66]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male       2916
Female     1817
Fe Male     155
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [67]:
dataset['Gender']=dataset['Gender'].str.replace('Fe Male','Female')

In [68]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male      2916
Female    1972
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Divorced      950
Single        916
Unmarried     682
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [69]:
dataset['MaritalStatus']=dataset['MaritalStatus'].str.replace('Single','Unmarried')

In [70]:
for col in categorical_cols:
    print('---------------------')
    print(dataset[col].value_counts())
    print('--------------------')

---------------------
TypeofContact
Self Enquiry       3444
Company Invited    1419
Name: count, dtype: int64
--------------------
---------------------
Occupation
Salaried          2368
Small Business    2084
Large Business     434
Free Lancer          2
Name: count, dtype: int64
--------------------
---------------------
Gender
Male      2916
Female    1972
Name: count, dtype: int64
--------------------
---------------------
ProductPitched
Basic           1842
Deluxe          1732
Standard         742
Super Deluxe     342
King             230
Name: count, dtype: int64
--------------------
---------------------
MaritalStatus
Married      2340
Unmarried    1598
Divorced      950
Name: count, dtype: int64
--------------------
---------------------
Designation
Executive         1842
Manager           1732
Senior Manager     742
AVP                342
VP                 230
Name: count, dtype: int64
--------------------


In [71]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
1897,201897,0,60.0,Company Invited,3,34.0,Small Business,Female,3,4.0,Standard,5.0,Married,5.0,0,1,1,0.0,Senior Manager,25266.0
982,200982,0,38.0,Company Invited,1,13.0,Salaried,Male,2,1.0,Basic,3.0,Married,2.0,0,3,0,0.0,Executive,17610.0
3764,203764,0,43.0,Company Invited,1,17.0,Large Business,Male,3,4.0,Basic,3.0,Married,5.0,0,4,1,1.0,Executive,21614.0
399,200399,0,31.0,Self Enquiry,1,6.0,Salaried,Male,3,4.0,Basic,5.0,Divorced,2.0,0,4,0,0.0,Executive,17218.0
639,200639,0,NaN,Self Enquiry,1,6.0,Large Business,Female,2,3.0,Basic,5.0,Divorced,3.0,0,4,1,0.0,Executive,18580.0


In [72]:
dataset['Age'] = dataset['Age'].fillna(dataset['Age'].median())
dataset['TypeofContact'] = dataset['TypeofContact'].fillna(dataset['TypeofContact'].mode()[0])
dataset['DurationOfPitch'] = dataset['DurationOfPitch'].fillna(dataset['DurationOfPitch'].median())
dataset['NumberOfFollowups'] = dataset['NumberOfFollowups'].fillna(dataset['NumberOfFollowups'].mode()[0])
dataset['PreferredPropertyStar'] = dataset['PreferredPropertyStar'].fillna(dataset['PreferredPropertyStar'].mode()[0])
dataset['NumberOfTrips'] = dataset['NumberOfTrips'].fillna(dataset['NumberOfTrips'].median())
dataset['NumberOfChildrenVisiting'] = dataset['NumberOfChildrenVisiting'].fillna(dataset['NumberOfChildrenVisiting'].mode()[0])
dataset['MonthlyIncome'] = dataset['MonthlyIncome'].fillna(dataset['MonthlyIncome'].median())


In [73]:
dataset.sample(5)

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
1976,201976,0,37.0,Self Enquiry,1,9.0,Salaried,Male,2,3.0,Standard,3.0,Married,2.0,0,1,1,0.0,Senior Manager,24434.0
2880,202880,1,34.0,Self Enquiry,1,17.0,Small Business,Male,3,6.0,Basic,3.0,Married,2.0,0,5,0,1.0,Executive,22086.0
3009,203009,0,24.0,Self Enquiry,1,17.0,Small Business,Male,4,3.0,Basic,3.0,Unmarried,3.0,0,2,1,2.0,Executive,22183.0
1933,201933,1,26.0,Self Enquiry,1,30.0,Large Business,Male,3,5.0,Basic,3.0,Unmarried,2.0,1,3,0,1.0,Executive,17340.0
315,200315,0,43.0,Company Invited,1,16.0,Salaried,Female,2,3.0,Basic,3.0,Unmarried,1.0,0,5,0,1.0,Executive,17455.0


In [74]:
dataset.isna().sum()

CustomerID                  0
ProdTaken                   0
Age                         0
TypeofContact               0
CityTier                    0
DurationOfPitch             0
Occupation                  0
Gender                      0
NumberOfPersonVisiting      0
NumberOfFollowups           0
ProductPitched              0
PreferredPropertyStar       0
MaritalStatus               0
NumberOfTrips               0
Passport                    0
PitchSatisfactionScore      0
OwnCar                      0
NumberOfChildrenVisiting    0
Designation                 0
MonthlyIncome               0
dtype: int64

In [75]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [76]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,Basic,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,Basic,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,3,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,1.0,Manager,26576.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,4,5.0,Basic,3.0,Unmarried,3.0,1,3,1,2.0,Executive,21212.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4,4.0,Standard,4.0,Married,7.0,0,1,1,3.0,Senior Manager,31820.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,3,4.0,Basic,3.0,Unmarried,3.0,0,5,0,2.0,Executive,20289.0


In [77]:
# Feature Engineerig

dataset['TotalNoOfPeople']=dataset['NumberOfPersonVisiting']+dataset['NumberOfChildrenVisiting']

In [78]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,...,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,...,3.0,Unmarried,1.0,1,2,1,0.0,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,4.0,...,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,4.0,...,3.0,Unmarried,7.0,1,3,0,0.0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,3.0,...,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,2,3.0,...,4.0,Divorced,1.0,0,5,1,0.0,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,3,5.0,...,4.0,Unmarried,2.0,1,1,1,1.0,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,4,5.0,...,3.0,Unmarried,3.0,1,3,1,2.0,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4,4.0,...,4.0,Married,7.0,0,1,1,3.0,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,3,4.0,...,3.0,Unmarried,3.0,0,5,0,2.0,Executive,20289.0,5.0


In [79]:
dataset.drop(['NumberOfChildrenVisiting','NumberOfPersonVisiting'],axis=1,inplace=True)

In [80]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [81]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4888 entries, 0 to 4887
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              4888 non-null   int64  
 1   ProdTaken               4888 non-null   int64  
 2   Age                     4888 non-null   float64
 3   TypeofContact           4888 non-null   object 
 4   CityTier                4888 non-null   int64  
 5   DurationOfPitch         4888 non-null   float64
 6   Occupation              4888 non-null   object 
 7   Gender                  4888 non-null   object 
 8   NumberOfFollowups       4888 non-null   float64
 9   ProductPitched          4888 non-null   object 
 10  PreferredPropertyStar   4888 non-null   float64
 11  MaritalStatus           4888 non-null   object 
 12  NumberOfTrips           4888 non-null   float64
 13  Passport                4888 non-null   int64  
 14  PitchSatisfactionScore  4888 non-null   

In [82]:
# Extract the number of numerical cols and categorical cols
categorical_cols=dataset.select_dtypes(include='O').columns
numerical_cols=dataset.select_dtypes(exclude='O').columns

In [83]:
categorical_cols

Index(['TypeofContact', 'Occupation', 'Gender', 'ProductPitched',
       'MaritalStatus', 'Designation'],
      dtype='object')

In [84]:
numerical_cols

Index(['CustomerID', 'ProdTaken', 'Age', 'CityTier', 'DurationOfPitch',
       'NumberOfFollowups', 'PreferredPropertyStar', 'NumberOfTrips',
       'Passport', 'PitchSatisfactionScore', 'OwnCar', 'MonthlyIncome',
       'TotalNoOfPeople'],
      dtype='object')

In [85]:
dataset

,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,0,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,1,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,1,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,1,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,1,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [86]:
# Divide the Dataset
X=dataset.drop('ProdTaken',axis=1)
y=dataset['ProdTaken']

In [87]:
X

,CustomerID,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
0,200000,41.0,Self Enquiry,3,6.0,Salaried,Female,3.0,Deluxe,3.0,Unmarried,1.0,1,2,1,Manager,20993.0,3.0
1,200001,49.0,Company Invited,1,14.0,Salaried,Male,4.0,Deluxe,4.0,Divorced,2.0,0,3,1,Manager,20130.0,5.0
2,200002,37.0,Self Enquiry,1,8.0,Free Lancer,Male,4.0,Basic,3.0,Unmarried,7.0,1,3,0,Executive,17090.0,3.0
3,200003,33.0,Company Invited,1,9.0,Salaried,Female,3.0,Basic,3.0,Divorced,2.0,1,5,1,Executive,17909.0,3.0
4,200004,36.0,Self Enquiry,1,8.0,Small Business,Male,3.0,Basic,4.0,Divorced,1.0,0,5,1,Executive,18468.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4883,204883,49.0,Self Enquiry,3,9.0,Small Business,Male,5.0,Deluxe,4.0,Unmarried,2.0,1,1,1,Manager,26576.0,4.0
4884,204884,28.0,Company Invited,1,31.0,Salaried,Male,5.0,Basic,3.0,Unmarried,3.0,1,3,1,Executive,21212.0,6.0
4885,204885,52.0,Self Enquiry,3,17.0,Salaried,Female,4.0,Standard,4.0,Married,7.0,0,1,1,Senior Manager,31820.0,7.0
4886,204886,19.0,Self Enquiry,3,16.0,Small Business,Male,4.0,Basic,3.0,Unmarried,3.0,0,5,0,Executive,20289.0,5.0


In [88]:
y

0       1
1       0
2       1
3       0
4       0
       ..
4883    1
4884    1
4885    1
4886    1
4887    1
Name: ProdTaken, Length: 4888, dtype: int64

In [89]:
# Train test Split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [90]:
X_train

,CustomerID,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,Designation,MonthlyIncome,TotalNoOfPeople
736,200736,48.0,Self Enquiry,1,10.0,Salaried,Male,4.0,Standard,3.0,Unmarried,1.0,0,5,1,Senior Manager,25999.0,4.0
1615,201615,30.0,Self Enquiry,1,11.0,Large Business,Female,3.0,Basic,5.0,Married,6.0,0,5,0,Executive,18204.0,3.0
336,200336,29.0,Self Enquiry,1,14.0,Salaried,Male,5.0,Basic,5.0,Divorced,2.0,1,3,1,Executive,17119.0,4.0
4526,204526,29.0,Self Enquiry,3,9.0,Small Business,Female,4.0,Deluxe,4.0,Married,3.0,1,3,1,Manager,23457.0,5.0
2665,202665,34.0,Self Enquiry,1,11.0,Small Business,Female,5.0,Basic,4.0,Divorced,8.0,0,4,0,Executive,21300.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,204426,28.0,Self Enquiry,1,10.0,Small Business,Male,5.0,Basic,3.0,Unmarried,2.0,0,1,1,Executive,20723.0,5.0
466,200466,41.0,Self Enquiry,3,8.0,Salaried,Female,3.0,Super Deluxe,5.0,Divorced,1.0,0,5,1,AVP,31595.0,4.0
3092,203092,38.0,Company Invited,3,28.0,Small Business,Female,4.0,Basic,3.0,Divorced,7.0,0,2,1,Executive,21651.0,5.0
3772,203772,28.0,Self Enquiry,3,30.0,Small Business,Female,5.0,Deluxe,3.0,Married,3.0,0,1,1,Manager,22218.0,5.0


In [91]:
y_test

144     0
79      0
2098    0
4738    0
2858    1
       ..
2570    1
3901    0
3364    0
3639    0
1962    0
Name: ProdTaken, Length: 1467, dtype: int64

In [ ]:

transformer = ColumnTransformer(
    transformers=[
        ('oneHotEncoder', OneHotEncoder(drop='first'), categorical_cols)
        ('Scaler',StandardScaler(),numerical_cols)
    ],
    remainder='passthrough'   # keeps non-categorical columns if any
)

# Fit and transform
X_train_transformed = transformer.fit_transform(X_train)

# Get column names
encoded_cols = transformer.get_feature_names_out()

# Convert to DataFrame with correct column names
X_train = pd.DataFrame(
    X_train_transformed,
    columns=encoded_cols,
    index=X_train.index
)


In [94]:
X_test_trasformed=transformer.transform(X_test)

In [93]:
X_train

,oneHotEncoder__TypeofContact_Self Enquiry,oneHotEncoder__Occupation_Large Business,oneHotEncoder__Occupation_Salaried,oneHotEncoder__Occupation_Small Business,oneHotEncoder__Gender_Male,oneHotEncoder__ProductPitched_Deluxe,oneHotEncoder__ProductPitched_King,oneHotEncoder__ProductPitched_Standard,oneHotEncoder__ProductPitched_Super Deluxe,oneHotEncoder__MaritalStatus_Married,...,remainder__CityTier,remainder__DurationOfPitch,remainder__NumberOfFollowups,remainder__PreferredPropertyStar,remainder__NumberOfTrips,remainder__Passport,remainder__PitchSatisfactionScore,remainder__OwnCar,remainder__MonthlyIncome,remainder__TotalNoOfPeople
736,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,1.0,10.0,4.0,3.0,1.0,0.0,5.0,1.0,25999.0,4.0
1615,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,11.0,3.0,5.0,6.0,0.0,5.0,0.0,18204.0,3.0
336,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,14.0,5.0,5.0,2.0,1.0,3.0,1.0,17119.0,4.0
4526,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,3.0,9.0,4.0,4.0,3.0,1.0,3.0,1.0,23457.0,5.0
2665,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,11.0,5.0,4.0,8.0,0.0,4.0,0.0,21300.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4426,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,10.0,5.0,3.0,2.0,0.0,1.0,1.0,20723.0,5.0
466,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,3.0,8.0,3.0,5.0,1.0,0.0,5.0,1.0,31595.0,4.0
3092,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.0,28.0,4.0,3.0,7.0,0.0,2.0,1.0,21651.0,5.0
3772,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,...,3.0,30.0,5.0,3.0,3.0,0.0,1.0,1.0,22218.0,5.0


In [96]:
X_test_trasformed

array([[0.0000e+00, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 1.9668e+04,
        3.0000e+00],
       [1.0000e+00, 0.0000e+00, 0.0000e+00, ..., 0.0000e+00, 2.0021e+04,
        4.0000e+00],
       [1.0000e+00, 0.0000e+00, 0.0000e+00, ..., 1.0000e+00, 2.1334e+04,
        3.0000e+00],
       ...,
       [0.0000e+00, 0.0000e+00, 0.0000e+00, ..., 1.0000e+00, 2.3122e+04,
        4.0000e+00],
       [1.0000e+00, 0.0000e+00, 1.0000e+00, ..., 1.0000e+00, 3.4057e+04,
        4.0000e+00],
       [1.0000e+00, 0.0000e+00, 1.0000e+00, ..., 0.0000e+00, 3.0402e+04,
        2.0000e+00]], shape=(1467, 27))

In [ ]:
# Now Both The features are Encoded Now We will Build the pipeline
transformer = ColumnTransformer(
    transformers=[
        ('oneHotEncoder', OneHotEncoder(drop='first'), categorical_cols),
        ('Scaler',StandardScaler(),numerical_cols)
    ],
    remainder='passthrough'   # keeps non-categorical columns if any
)

def checkScores(models):
    cv=StratifiedKFold(shuffle=True,random_state=42)
    for m in models:
        pipe=Pipeline([
            ('preprocessing',transformer),
            ('model',m)
        ])
        pipe.fit(X_train_transformed,y_train)
        print(f'----------------------{m} Model Trained-------------------')
        y_predicted=cross_val_predict(estimator=pipe,X=X_train_transformed,y=y_train,cv=cv)
        
        print(f'----->{m} Model Metricess On traing data')
        print(f'Accuracy Score: {accuracy_score(y_pred=y_predicted,y_true=y_train)}')
        print(f'Confusion Matrix:\n {confusion_matrix(y_pred=y_predicted,y_true=y_train)}')
        print(f'Classification report: \n {classification_report(y_pred=y_predicted,y_true=y_train)}')

        y_predicted=cross_val_predict(estimator=pipe,X=X_test_trasformed,y=y_test,cv=cv)
        
        print(f'---------->{m} Model Metricess On traing data')
        print(f'Accuracy Score: {accuracy_score(y_pred=y_predicted,y_true=y_test)}')
        print(f'Confusion Matrix:\n {confusion_matrix(y_pred=y_predicted,y_true=y_test)}')
        print(f'Classification report: \n {classification_report(y_pred=y_predicted,y_true=y_test)}')

In [114]:
from sklearn.linear_model import LogisticRegression
logistic=LogisticRegression(max_iter=10000)
checkScores([logistic])

----------------------LogisticRegression(max_iter=10000) Model Trained-------------------


d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/pre

----->LogisticRegression(max_iter=10000) Model Metricess On traing data
Accuracy Score: 0.8430283542823735
Confusion Matrix:
 [[2688   87]
 [ 450  196]]
Classification report: 
               precision    recall  f1-score   support

           0       0.86      0.97      0.91      2775
           1       0.69      0.30      0.42       646

    accuracy                           0.84      3421
   macro avg       0.77      0.64      0.67      3421
weighted avg       0.83      0.84      0.82      3421

---------->LogisticRegression(max_iter=10000) Model Metricess On traing data
Accuracy Score: 0.83640081799591
Confusion Matrix:
 [[1150   43]
 [ 197   77]]
Classification report: 
               precision    recall  f1-score   support

           0       0.85      0.96      0.91      1193
           1       0.64      0.28      0.39       274

    accuracy                           0.84      1467
   macro avg       0.75      0.62      0.65      1467
weighted avg       0.81      0.84      0.8

d:\Udemy\DataScience\ML-DL-NLP\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
